# GSPO


## 参考

[Group Sequence Policy Optimization](https://arxiv.org/pdf/2507.18071)  
[HF](https://huggingface.co/docs/trl/grpo_trainer)  
[verl](https://swift.readthedocs.io/en/latest/Instruction/GRPO/AdvancedResearch/GSPO.html)

## GSPO
- 提出GRPO在公式层面上的设计缺陷：its objective is ill-posed，导致在训练需要long-response的任务上出现崩溃，并且是不可逆的。
    - insight：在重要性采样：
    $$\mathbb{E}_{z\sim\pi_{\text{tar}}}[f(z)]
    = \mathbb{E}_{z\sim\pi_{\text{beh}}}\!\left[\frac{\pi_{\text{tar}}(z)}{\pi_{\text{beh}}(z)}f(z)\right]$$
    - 对于每一个token估计权重(sampleing ratio)随之而来的问题是：
        - ratio没有统计意义，反而造成了高方差
        - 在长序列累计后引发崩溃（ is exacerbated by the clipping mechanism）
    - 核心观点：the unit of optimization objective should match the unit of reward. 

- 核心转变：
    对于原来的token级别转变为序列级别，
    $$\frac{\pi_{\theta}(y_{i, t})}{\pi_{\text{old}}(y_{i, t})} \rightarrow \frac{\pi_{\theta}(y_{i}\mid x)}{\pi_{\theta_{\text{old}}}(y_{i}\mid x)}$$
    - meaning：衡量了完整的respomse在当前策略 $\pi_{\theta}$ 的可能性，比在就策略 $\pi_{\text{old}}$ 下高或低多少

## GSPO的
- 序列级别的优化目标为：

    $$ J_{\text{GSPO}}(\theta) = \mathbb{E}_{x\sim\mathcal{D},\,(y_i)_{i=1}^{G}\sim\pi_{\theta_{\text{old}}}} \sum_{i=1}^{G} \min\!\bigl(s_{i}(\theta)\hat{A}_{i},\; \text{clip}\bigl(s_{i}(\theta),1-\varepsilon,1+\varepsilon\bigr)\hat{A}_{i}\bigr) $$

    GSAE(group-based advantage estimaion):

    $$ \hat{A}_{i} = \frac{r(x,y_{i})- \text{mean}({\{ r(x, y_i)\}_{i=1}^{G}})}{\mathrm{std}\bigl((r(x,y_{j}))_{i=1}^{G}\bigr)} $$

    以及基于序列的重要性比（importance ratio）：

    $$ s_{i}(\theta) 
    = \left(\frac{\pi_{\theta}(y_{i}\mid x)}{\pi_{\theta_{\text{old}}}(y_{i}\mid x)}\right)^{\frac{1}{|y_i|}} 
    = \left(\prod_{t=1}^{|y_{i}|} \frac{\pi_{\theta}(y_{i,t}\mid x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}\mid x,y_{i,<t})} \right)^{\frac{1}{|y_i|}}\
    =\text{exp} \left(\frac{1}{|y_i|} \sum_{t = 1}^{|y_i|} \log{\frac{\pi_{\theta}(y_{i,t}\mid x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}\mid x,y_{i,<t})}} \right)$$


- 对于梯度部分：
    $$
    \begin{aligned}
    \nabla_{\theta} \mathcal{J}_{\text{GSPO}}(\theta) 

    &= \mathbb{E}_{x \sim \mathcal{D}, \{y_i\}_{i=1}^{G} \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)} \left[ \frac{1}{G} \sum_{i=1}^{G} \left( \frac{\pi_{\theta}(y_i \mid x)}{\pi_{\theta_{\text{old}}}(y_i \mid x)} \right)^{\frac{1}{|y_i|}} \hat{A}_i \cdot \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \nabla_{\theta} \log \pi_{\theta}(y_{i,t} \mid x, y_{i,<t}) \right].
    \end{aligned}
    $$


    $$
    \begin{aligned}
    \nabla_{\theta} \mathcal{J}_{\text{GRPO}}(\theta)

    &= \mathbb{E}_{x \sim \mathcal{D}, \{y_i\}_{i=1}^{G} \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)} \left[ \frac{1}{G} \sum_{i=1}^{G} \hat{A}_i \cdot \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \frac{\pi_{\theta}(y_{i,t} \mid x, y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t} \mid x, y_{i,<t})} \nabla_{\theta} \log \pi_{\theta}(y_{i,t} \mid x, y_{i,<t}) \right].
    \end{aligned}
    $$

    - 本质：是否对一个句子里的tokens进行加权
     - 对于importance weight GRPO的加权可能从 $( 0, 1-\epsilon](\hat{A}>0) \text{and} [1-\epsilon, + \infty)(\hat{A}<0)$

- token级别的GSPO

    在一些任务上需要更细的颗粒度（token），使得每个评价token对整体sequence的贡献。
    $$\mathcal{J}_{\text{GSPO-token}}(\theta) = \mathbb{E}_{x \sim \mathcal{D}, \{y_i\}_{i=1}^{G} \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)} \left[ \frac{1}{G} \sum_{i=1}^{G} \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \min \left( s_{i,t}(\theta) \hat{A}_{i,t}, \; \text{clip}\left(s_{i,t}(\theta), 1-\varepsilon, 1+\varepsilon\right) \hat{A}_{i,t} \right) \right],$$
    其中：
    $$
    s_{i,t}(\theta) = \text{sg}\left[s_{i}(\theta)\right] \cdot \frac{\pi_{\theta}(y_{i,t} \mid x, y_{i,<t})}{\text{sg}\left[\pi_{\theta}(y_{i,t} \mid x, y_{i,<t})\right]},
    $$
    细节见[paper](https://arxiv.org/abs/2507.18071)

## Change in Routing play
![fixed](./img/relplay%20MoE.png)
![GRPO MoE](./img/GRPO%20routing%20paly.png)


- GRPO 使用routing play进行稳定训练（当先训练使用前几次的router，先不改变路由策略）
    - 作用：防止collapse进行稳定训练，
    - 缺点：占用额外内存，可能限制 model 容量

- GSPO 对于整个 sequence 的 likelihood，对 token 的敏感度降低